In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
import torch
import pandas as pd
import pickle
import numpy as np
from lightning_modelling.common_path import DATASET_PATH, MODELS_PATH
from lightning_modelling.deter_architecture import Unet, FullyConnectedNet_1d
from lightning_modelling.models import XGBoostModel, LogisticRegressionModel, GAMModel
from lightning_modelling.dataset import create_train_test
from lightning_modelling.plots import LightningPlotMultiModel

## Load data

In [ ]:
ALL_YEARS = list(range(2008, 2024))
HELD_OUT_YEARS = [2008, 2015, 2023]
ALL_LOYO_YEARS = [year for year in ALL_YEARS if year not in HELD_OUT_YEARS]
TRAIN_YEARS = ALL_LOYO_YEARS
TEST_YEARS = HELD_OUT_YEARS
SCALER_PATH = os.path.join(DATASET_PATH, "scaler", "scaler_full.pkl")

print("Creating the dataset..............")
_, _, TEST_DATASET = create_train_test(DATASET_PATH, TRAIN_YEARS, TEST_YEARS, scaler_path=SCALER_PATH)
TEST_DATASET.metadata_csv["year"] = pd.to_datetime(TEST_DATASET.metadata_csv["date"]).dt.year

extreme_days = pd.read_csv(DATASET_PATH / "extreme_days_top_0.05.csv")
all_extremes_metadata = TEST_DATASET.metadata_csv[TEST_DATASET.metadata_csv["date"].isin(extreme_days["date"])]
test_extremes = all_extremes_metadata[all_extremes_metadata["year"].isin(TEST_YEARS)]

## Load models

In [ ]:
CHANNELS = [16, 32, 64]
NUM_RESIDUAL_LAYERS = 2
RECALIBRATION = 'platt_scaling'

# Initialize unet
unet = Unet(
    channels=CHANNELS,
    num_residual_layers=NUM_RESIDUAL_LAYERS,
    name="unet",
    recalibration_method=RECALIBRATION,
)

unet.load_state_dict(torch.load(MODELS_PATH / 'unet.pth', map_location=torch.device('cpu')))
unet.eval()

xgb_name = "xgb"
xgb_model = pickle.load(open(MODELS_PATH / "xgb.pkl", "rb"))
xgb = XGBoostModel(model=xgb_model, name=xgb_name, remove_vars=None)


gam_name = "gam"
gam_model = pickle.load(open(MODELS_PATH / "gam.pkl", "rb"))
gam = GAMModel(model=gam_model, name=gam_name, remove_vars=None)


mlp_name = "mlp"
HIDDEN_DIMS = [16, 32, 16]

mlp = FullyConnectedNet_1d(
    name=mlp_name,
    save_path=None,
    hidden_dims=HIDDEN_DIMS,
    recalibration_method=RECALIBRATION,
    removed_features=[],
)

mlp.load_state_dict(torch.load(MODELS_PATH / "mlp.pth", map_location=torch.device('cpu')))
mlp.eval()

logreg_name = "logreg"
logreg_model = pickle.load(open(MODELS_PATH / "log_reg.pkl", "rb"))
logreg = LogisticRegressionModel(model=logreg_model, name=logreg_model, remove_vars=None)

models = [logreg, gam, xgb, mlp, unet]

## Plot extreme days

In [ ]:
test_extremes.head()

In [ ]:
dates_to_plot = test_extremes["date"].tolist()[10:14]

for i, sample in enumerate(TEST_DATASET):
    date = test_extremes.iloc[i]["date"]
    if date not in dates_to_plot:
        continue

    obs = (sample[:, -1, :, :] >= 2).float()
    obs = obs.sum(axis=0).numpy()
    data = obs[np.newaxis, :, :]
    for model in models:
        preds = model(sample[:, :-1, :, :])
        if isinstance(preds, torch.Tensor):
            preds = preds.sum(axis=0).detach().numpy()
        data = np.concatenate([data, preds[np.newaxis, :, :]], axis=0)

    full_rmses = []
    conditionned_rmses = []
    for i in range(1, data.shape[0]):
        full_rmses.append(np.sqrt(np.mean((data[0] - data[i])**2)))    # RMSE across the whole map
        conditionned_rmses.append(np.sqrt(np.mean((data[0][data[0] > 0] - data[i][data[0] > 0])**2)))  # RMSE for observations > 0
    max_hours = data.max()
    levels = np.linspace(0, int(np.ceil(max_hours)), int(round(max_hours)) + 1)
    event_plot = LightningPlotMultiModel(
        lightning_hours=data,
        metadata_json=TEST_DATASET.metadata_json,
        title=f"{date} event",
        save_path=None,
        file_name=f"{date}.png",
        levels=levels,
        full_errors=full_rmses,
        conditionned_errors=conditionned_rmses
    )
    event_plot.show()
    # event_plot.save()